In [ ]:
import boto3
import botocore
import functools
from IPython.core.display import display, HTML
from iterdub import iterdub as ib
from iterpop import iterpop as ip
import itertools as it
import json
import matplotlib
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
import seaborn as sns
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyanalysis import calc_loglikelihoods_by_num_sets
from dishpylib.pyanalysis import count_hands_with_k_or_more_sets
from dishpylib.pyanalysis import count_hands_without_k_or_more_sets
from dishpylib.pyanalysis import estimate_interpolation_complexity
from dishpylib.pyanalysis import calc_loglikelihoods_over_set_sizes
from dishpylib.pyhelpers import get_env_context
from dishpylib.pyhelpers import get_git_revision_hash
from dishpylib.pyhelpers import make_timestamp
from dishpylib.pyhelpers import NumpyEncoder
from dishpylib.pyhelpers import preprocess_competition_fitnesses
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2025-09-13-interspersed-nopouts"


In [ ]:
import pandas as pd
from scipy import stats

def fit_control_t_distns(control_df):

    na_rows = control_df['Fitness Differential Focal'].isna()
    assert all( control_df[ na_rows ]['Population Extinct'] )
    print(na_rows.sum())
    control_df['Fitness Differential Focal'].fillna(0, inplace=True,)

    res = []
    for series in control_df['Competition Series'].unique():

        series_df = control_df[ control_df['Competition Series'] == series ]

        # legacy data was mixed inside of the variant_df
        # wt_vs_wt_df = series_df.groupby('Competition Repro').filter(
        #     lambda x: (x['genome variation'] == 'master').all()
        # ).groupby('Competition Repro').first().reset_index()

        # fit a t distribution to the control data
        # df is degrees of freedom
        df, loc, scale = stats.t.fit( series_df['Fitness Differential Focal'] )


        res.append({
            'Series' : series,
            'Fit Degrees of Freedom' : df,
            'Fit Loc' : loc,
            'Fit Scale' : scale,
        })

    return pd.DataFrame(res)


In [ ]:
import boto3
import botocore
import functools
import pandas as pd


@functools.lru_cache
def get_control_t_distns( bucket, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = [*bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/control-competitions-focalbb-phenotypeneutral/stage=3+what=collated/stint={stint}',
    )][:1]

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )

    res = fit_control_t_distns(control_df[
        control_df["Root ID"] == 0
    ].copy())
    return res


In [ ]:
import functools
from iterpop import iterpop as ip
from scipy import stats


def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: stats.t.cdf(
            row["Fitness Differential Focal"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 40
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 40)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49')

dfs = []
for stint in range(101):
    print(f'stint: {stint}')
    series_profile, = bucket_handle.objects.filter(
        Prefix=f'endeavor=16/bioticbackground-noncritical-phenotypeneutral-nopinterspersion-competitions/stage=8+what=collated/stint={stint}/',
    )

    control_fits_df = get_control_t_distns('prq49', 16, stint)
    df = pd.read_csv(
        f's3://prq49/{series_profile.key}',
    )
    df = df[df["Competition Series"] == 16005]
    df = df[df["genome morph"] != "wildtype"].copy()
    df = df[df["genome a"] == "genome"].copy()

    print(f'stint: {stint}, df length: {len(df)}')
    df["Stint"] = stint
    dfdigest = '{:x}'.format( hash_pandas_object( df ).sum() )
    print(dfdigest)
    df = preprocess_competition_fitnesses(df, control_fits_df)
    dfs.append(df)


In [ ]:
df = pd.concat(dfs)


In [ ]:
dfx = df[df["genome variation"] != "master"]


In [ ]:
dfx


In [ ]:
dfz = dfx.groupby(["Stint"]).sum(numeric_only=True).reset_index(drop=False)
dfz


In [ ]:
sns.lineplot(
    data=dfz,
    x="Stint",
    y="Is More Fit",
)
sns.lineplot(
    data=dfz,
    x="Stint",
    y="Is Less Fit",
)
# Add small text labels for each point
for x, y in zip(dfz["Stint"], dfz["Is Less Fit"]):
    plt.text(x, y, str(x), fontsize=8, ha='right', va='bottom')

# sns.lineplot(
#     data=dfz,
#     x="Stint",
#     y="Is Neutral",
# )


In [ ]:
sns.regplot(
    data=dfz[
        (dfz["Stint"] > 40)
        & (dfz["Stint"] < 100)
    ],
    x="Stint",
    y="Is Less Fit",
    scatter=True,
)


In [ ]:
ys1 = dfz.loc[
    (dfz["Stint"] > 0)
    & (dfz["Stint"] < 90),
    "Is Less Fit",
]
ys2 = dfz.loc[
    (dfz["Stint"] > 0)
    & (dfz["Stint"] < 90),
    "Stint",
]

# Parametric: Pearson correlation
pearson_corr, pearson_p = stats.pearsonr(ys1, ys2)
print(f"Pearson correlation: r={pearson_corr:.3f}, p={pearson_p:.3g}")

# Nonparametric: Spearman correlation
spearman_corr, spearman_p = stats.spearmanr(ys1, ys2)
print(f"Spearman correlation: r={spearman_corr:.3f}, p={spearman_p:.3g}")


In [ ]:
sns.regplot(
    data=dfz[
        (dfz["Stint"] < 40)
    ],
    x="Stint",
    y="Is Less Fit",
    scatter=True,
)


In [ ]:
ys1 = dfz.loc[
    (dfz["Stint"] < 40),
    "Is Less Fit",
]
ys2 = dfz.loc[
    (dfz["Stint"] < 40),
    "Stint",
]

# Parametric: Pearson correlation
pearson_corr, pearson_p = stats.pearsonr(ys1, ys2)
print(f"Pearson correlation: r={pearson_corr:.3f}, p={pearson_p:.3g}")

# Nonparametric: Spearman correlation
spearman_corr, spearman_p = stats.spearmanr(ys1, ys2)
print(f"Spearman correlation: r={spearman_corr:.3f}, p={spearman_p:.3g}")


In [ ]:
sns.regplot(
    data=dfz[
        (dfz["Stint"] > 30)
        & (dfz["Stint"] < 85)
    ],
    x="Stint",
    y="Is Less Fit",
    scatter=True,
)
